# Week 12: File I/O & CSV — PHASE 3: Build a checkable report

*📚 Computer Programming I · ⏱️ 5 Hours · 👨‍🏫 Dr. Arif Solmaz*

## Save a report someone can check

We can now read and validate readings. A result that exists only in a running notebook is hard for another person to review tomorrow.

We will read a small sensor CSV and write accepted records to a separate file. For every rejected record, we keep the source line and the reason.

**Try this first — before code.** Sketch three labelled files: raw input, cleaned readings and rejection record. Where would another engineer look to explain a missing reading?

**Why this week's tool?** Reading and writing files keeps information after the run ends. CSV gives the records a shared structure. Reopening the saved output shows what was really written.

**By the end.** Follow one accepted and one rejected record from the raw file to the saved outputs. Never overwrite the original evidence.


## Explain the program: A saved file must be readable again

Writing text to a file keeps it after the program ends. A good report keeps enough structure to be read back with the same meaning. Formatting, separators, quoting and file mode all affect that.

**Draw or trace.** Draw memory → writer → file bytes → reader → rebuilt records. Put a sensor label that contains a comma in one field. This shows the difference between simple splitting and CSV parsing.

**Predict before running.** Will splitting every line at every comma correctly rebuild a quoted field that contains a comma? What happens when a second run uses write mode?

<details><summary>Trace and explanation — after your prediction</summary>

1. A CSV-aware writer quotes fields when needed. A CSV-aware reader rebuilds them.
2. A plain split on commas treats the comma inside a quoted field as one more separator.
3. Write mode replaces an existing file. Append mode adds content, and careless use can duplicate rows or headers.

Test a write-then-read round trip. Close the file, then inspect it. Equal rebuilt records show that the file stored the data correctly. They do not prove that the original measurements were right.

</details>

**Change one thing.** Run the export twice into a temporary file. Decide whether you want replacement or accumulation. Then check for duplicated headers.

**Türkçe:** Dosyaya yazmak veriyi saklar. CSV tırnaklaması ve dosya modu anlamı korumalı; yazdığını yeniden okuyarak denetle.


<details><summary>Learning objectives</summary>

## 🎯 Learning Objectives

By the end of this week, you will be able to:

- Understand why programs need to read and write files
- Open, read, and close files using Python's `open()` function and the `with` statement
- Write data to text files using `write()` and `writelines()`
- Distinguish between file modes (`'r'`, `'w'`, `'a'`, `'r+'`)
- Process text files line by line
- Understand the CSV (Comma-Separated Values) format
- Read and parse CSV data using string methods
- Write data in CSV format
- Build a practical program that reads, processes, and writes data files

</details>


<details><summary>Class participation and assessment</summary>

---
## 🤝 Engineering Learning Contract

- **Professional relevance:** examples and core exercises model the data, sensing, automation, numerical, and decision tasks used in engineering.
- **Interaction:** predict before running, compare reasoning with a partner, and ask whenever a step is unclear; scheduled checkpoints guarantee question time.
- **Assessment alignment:** worked examples and Core Exercises 1–8 rehearse the same reasoning operations used on exams—trace, implement, debug, interpret, and justify—while exam values and contexts may change.
- **Assessment:** Assessment consists only of the midterm exam (50%) and final exam (50%). Weekly notebooks, exercises, projects, demonstrations and presentations are ungraded practice; no weekly submission is required.

</details>


---
## 📦 Setup

Run this cell first to load the required packages.

In [1]:
import os

<details><summary>Class schedule and checkpoints</summary>

---
## 🧭 Five-Hour Class Roadmap

This notebook is designed for one five-hour class with four short breaks.

| Target | Activity |
|---|---|
| 00:00–00:55 | Concepts and examples → Checkpoint 1 |
| 00:55–01:05 | Break |
| 01:05–01:55 | Concepts and examples → Checkpoint 2 |
| 01:55–02:05 | Break |
| 02:05–02:55 | Concepts and examples → Checkpoint 3 |
| 02:55–03:05 | Break |
| 03:05–03:55 | Concepts and examples → Checkpoint 4 |
| 03:55–04:05 | Break |
| 04:05–04:45 | Core Practice (Exercises 1–8) → Checkpoint 5 |
| 04:45–05:00 | Review and retry failed checks |

Concept checkpoints compare your predictions with an expected answer. Checkpoint 5 is your practice reflection. No grading submission is sent by these tools. Save the notebook to keep your work. Exercises 9 and above are optional extensions.

</details>


In [ ]:
#@title Study tools — run this cell once (the code is hidden; you do not need to read it)
# These small tools give local study feedback.
_checkpoint_results = {}

def check_answer(number, answer, expected, explanation):
    actual = str(answer).strip().lower().replace(" ", "")
    target = str(expected).strip().lower().replace(" ", "")
    correct = actual == target
    _checkpoint_results[int(number)] = ("Concept check", int(correct), 1)
    if correct:
        print(f"Checkpoint {number}: correct. {explanation}")
    elif not str(answer).strip():
        print(f"Checkpoint {number}: enter your prediction, then run again.")
    else:
        print(f"Checkpoint {number}: review the example and try again.")
    return correct

def record_checkpoint(number, checks):
    """Report each concrete concept check used by the introductory notebook."""
    passed = sum(bool(correct) for _, correct in checks)
    _checkpoint_results[int(number)] = ("Concept checks", passed, len(checks))
    print(f"Checkpoint {number}: {passed}/{len(checks)} concept checks match.")
    for label, correct in checks:
        print(("OK: " if correct else "Review: ") + label)
    return passed, len(checks)

def exercise_checkpoint(number, practiced, expected=8):
    """Summarize an explicit self-report; this does not grade your code."""
    if not isinstance(practiced, (list, tuple, set)):
        _checkpoint_results.pop(int(number), None)
        print("Use a list of exercise numbers, for example [1, 2].")
        return 0, expected
    if any(type(item) is not int or not 1 <= item <= expected for item in practiced):
        _checkpoint_results.pop(int(number), None)
        print(f"Use whole exercise numbers from 1 to {expected}.")
        return 0, expected
    done = set(practiced)
    _checkpoint_results[int(number)] = ("Practice self-report", len(done), expected)
    print(f"Practice self-report: {len(done)}/{expected} core exercises reviewed.")
    print("This is your reflection, not a correctness score or a grade.")
    remaining = [str(i) for i in range(1, expected + 1) if i not in done]
    if remaining:
        print("Still to review:", ", ".join(remaining))
    print("For each exercise: test the result, explain the steps, then compare with the worked solution.")
    return len(done), expected

def show_progress_summary():
    print("\nMy study feedback (this runtime)")
    for number in range(1, 6):
        if number in _checkpoint_results:
            kind, count, total = _checkpoint_results[number]
            print(f"{number}. {kind}: {count}/{total}")
        else:
            print(f"{number}. Not run yet")
    print("These checks send no grading submission. Save your notebook to keep your work.")

print("Local study tools ready.")


## Joining this lesson: records and returned pairs

Recall a small record: `student = {"name": "Elif", "scores": [85, 92]}`.
`student["name"]` reads a key; `student["average"] = 88.5` adds a value;
`"scores" in student` checks whether the key exists. To display the entries, use
`for key, value in student.items(): print(key, value)`.

A function can `return True, "OK"`. Its caller writes `valid, reason = validate(...)`
to unpack the returned tuple. Keep the order consistent: first the decision, then
the explanation. Dictionary keys name fields; tuple positions preserve a short,
agreed order. Dictionaries were introduced in Week 7 and returned pairs in Weeks 9–10.

**Türkçe:** Sözlük anahtarı alanın adıdır; demet açma iki sonucu sırayla alır.
Bir kaydı okumadan önce gerekli alanların bulunduğunu kontrol edin.


---
## Part 1: Why File I/O?

So far, all data in our programs has been **temporary**. When the program ends, everything is lost. Think about it:

| Without File I/O | With File I/O |
|:---|:---|
| Data disappears when program ends | Data is **saved** to disk |
| User must re-enter data every time | Data can be **loaded** next time |
| Cannot process large datasets | Can read **thousands** of records from a file |
| Cannot share data between programs | Files can be **shared** between programs |

**File I/O** (Input/Output) lets our programs:
- **Read** data from files (input)
- **Write** data to files (output)
- **Persist** information between program runs

> 💡 **Note:** Think of a file like a notebook. You can write notes in it, close it, and open it later to read what you wrote. That's exactly what file I/O does for programs!

---
## Part 2: Opening and Reading Files

Python uses the built-in `open()` function to work with files. The basic syntax is:

```python
file = open(filename, mode)
```

Let's start by creating a sample file in Google Colab, then reading it.

### Creating a File with `%%writefile`

In Google Colab, we can create files using the `%%writefile` magic command. This writes the cell contents to a file.

**Figure 2.1: Creating a text file with `%%writefile`**

In [3]:
%%writefile greeting.txt
Hello, welcome to Python!
File I/O is very useful.
This is the third line.
Learning to read files is fun.

Writing greeting.txt


### Reading the Entire File with `read()`

**Figure 2.2: Reading an entire file with `read()`**

In [4]:
# Open the file for reading
file = open("greeting.txt", "r")

# Read the entire content
content = file.read()

# Close the file
file.close()

print(content)

Hello, welcome to Python!
File I/O is very useful.
This is the third line.
Learning to read files is fun.



### Reading Line by Line with `readline()`

**Figure 2.3: Reading one line at a time with `readline()`**

In [5]:
file = open("greeting.txt", "r")

line1 = file.readline()  # Reads first line
line2 = file.readline()  # Reads second line

print("First line:", line1.strip())
print("Second line:", line2.strip())

file.close()

First line: Hello, welcome to Python!
Second line: File I/O is very useful.


> 💡 **Note:** `readline()` includes the newline character `\n` at the end. Use `.strip()` to remove it.

### Reading All Lines as a List with `readlines()`

**Figure 2.4: Reading all lines into a list with `readlines()`**

In [6]:
file = open("greeting.txt", "r")

lines = file.readlines()  # Returns a list of lines

file.close()

print("Type:", type(lines))
print("Number of lines:", len(lines))
print("Lines:", lines)

Type: <class 'list'>
Number of lines: 4
Lines: ['Hello, welcome to Python!\n', 'File I/O is very useful.\n', 'This is the third line.\n', 'Learning to read files is fun.\n']


### The `with` Statement (Recommended!)

Forgetting to close a file can cause problems. The `with` statement **automatically closes** the file when done.

**Figure 2.5: Using the `with` statement for safe file handling**

In [7]:
# The 'with' statement automatically closes the file
with open("greeting.txt", "r") as file:
    content = file.read()
    print(content)

# File is automatically closed here
print("File is closed:", file.closed)

Hello, welcome to Python!
File I/O is very useful.
This is the third line.
Learning to read files is fun.

File is closed: True


> 💡 **Note:** Always prefer the `with` statement over manually opening and closing files. It's safer and cleaner!

---
### ⏱️ Checkpoint 1 of 5 — Resources (target 00:55)

Which statement automatically closes an opened file?

Enter a short answer in the next cell and run it. Retry after reviewing the
preceding examples if needed.


In [8]:
checkpoint_1_answer = ""  # enter your answer
check_answer(
    1, checkpoint_1_answer, 'with',
    'A `with` context manages cleanup.',
)


Checkpoint 1: enter your prediction, then run again.


False

---
## Part 3: Writing to Files

To write data to a file, open it in write mode (`'w'`) or append mode (`'a'`).

### Writing with `write()`

**Figure 3.1: Writing text to a file**

In [9]:
# Write mode ('w') creates a new file or overwrites existing
with open("output.txt", "w") as file:
    file.write("Hello from Python!\n")
    file.write("This is line 2.\n")
    file.write("This is line 3.\n")

# Verify by reading it back
with open("output.txt", "r") as file:
    print(file.read())

Hello from Python!
This is line 2.
This is line 3.



> 💡 **Note:** `write()` does **not** add a newline automatically. You must include `\n` yourself!

### Writing Multiple Lines with `writelines()`

**Figure 3.2: Writing a list of lines with `writelines()`**

In [10]:
lines = ["Apple\n", "Banana\n", "Cherry\n", "Date\n"]

with open("fruits.txt", "w") as file:
    file.writelines(lines)

# Verify
with open("fruits.txt", "r") as file:
    print(file.read())

Apple
Banana
Cherry
Date



### Appending to a File with `'a'` Mode

**Figure 3.3: Appending data to an existing file**

In [11]:
# Append mode ('a') adds to the end of the file
with open("fruits.txt", "a") as file:
    file.write("Elderberry\n")
    file.write("Fig\n")

# Verify - now has 6 fruits
with open("fruits.txt", "r") as file:
    print(file.read())

Apple
Banana
Cherry
Date
Elderberry
Fig



---
## Part 4: File Modes

Here is a summary of all important file modes:

| Mode | Description | File Exists? | File Doesn't Exist? |
|:----:|:---|:---|:---|
| `'r'` | Read only | Opens file | **Error!** |
| `'w'` | Write only | **Overwrites** file | Creates new file |
| `'a'` | Append only | Adds to end | Creates new file |
| `'r+'` | Read and write | Opens file | **Error!** |

> 💡 **Note:** Be very careful with `'w'` mode! It will **erase** all existing content in the file. If you want to add to a file, use `'a'` (append) mode instead.

**Figure 4.1: Demonstrating the difference between `'w'` and `'a'`**

In [12]:
# Write mode OVERWRITES the file
with open("demo.txt", "w") as f:
    f.write("First write\n")

with open("demo.txt", "w") as f:  # This ERASES previous content!
    f.write("Second write\n")

with open("demo.txt", "r") as f:
    print("After 'w' mode twice:")
    print(f.read())  # Only "Second write"

# Append mode ADDS to the file
with open("demo2.txt", "w") as f:
    f.write("First write\n")

with open("demo2.txt", "a") as f:  # This ADDS to existing content
    f.write("Second write\n")

with open("demo2.txt", "r") as f:
    print("After 'w' then 'a':")
    print(f.read())  # Both lines!

After 'w' mode twice:
Second write

After 'w' then 'a':
First write
Second write



---
### ⏱️ Checkpoint 2 of 5 — Modes (target 01:55)

Which file mode appends without erasing existing content?

Enter a short answer in the next cell and run it. Retry after reviewing the
preceding examples if needed.


In [13]:
checkpoint_2_answer = ""  # enter your answer
check_answer(
    2, checkpoint_2_answer, 'a',
    'Append mode writes at the end.',
)


Checkpoint 2: enter your prediction, then run again.


False

---
## Part 5: Working with Text Files

A very common pattern is to **read a file line by line** and process each line. You can iterate over a file object directly in a `for` loop.

**Figure 5.1: Creating a data file for processing**

In [14]:
%%writefile students.txt
Elif Demir 85
Burak Yilmaz 92
Zeynep Kaya 78
Mert Ozturk 95
Ayse Celik 88

Writing students.txt


**Figure 5.2: Reading and processing a file line by line**

In [15]:
with open("students.txt", "r") as file:
    for line in file:          # Iterate line by line
        line = line.strip()    # Remove newline
        parts = line.split()   # Split by whitespace
        
        first_name = parts[0]
        last_name = parts[1]
        grade = int(parts[2])
        
        if grade >= 90:
            status = "Excellent"
        elif grade >= 80:
            status = "Good"
        else:
            status = "Needs Improvement"
        
        print(f"{first_name} {last_name}: {grade} - {status}")

Elif Demir: 85 - Good
Burak Yilmaz: 92 - Excellent
Zeynep Kaya: 78 - Needs Improvement
Mert Ozturk: 95 - Excellent
Ayse Celik: 88 - Good


> 💡 **Note:** Iterating over a file with `for line in file:` is memory-efficient because it reads one line at a time, rather than loading the entire file into memory.

---
## Part 6: CSV Concept

**CSV** stands for **Comma-Separated Values**. It is one of the simplest and most common formats for storing tabular data.

A CSV file looks like this:
```
name,age,city
Elif,21,Istanbul
Burak,22,Ankara
Zeynep,20,Izmir
```

**Key characteristics:**
- Each line is a **row** (record)
- Values in each row are separated by **commas**
- The first row is usually the **header** (column names)
- Each row has the **same number** of values

| Feature | CSV | Excel |
|:---|:---|:---|
| Format | Plain text | Binary |
| Can open with | Any text editor | Needs Excel/similar |
| File size | Small | Larger |
| Easy to process | Yes, with Python | Needs special library |

> 💡 **Note:** Although Python has a `csv` module, at this level we will parse CSV files using simple string methods like `split(',')`. This helps you understand what's happening behind the scenes!

**Scope of this simple parser:** our examples contain no quoted commas or embedded newlines. `split(",")` is enough for these teaching files. General CSV may contain quoted fields; Python’s standard `csv` module handles those rules.


---
### ⏱️ Checkpoint 3 of 5 — CSV (target 02:55)

Which standard-library module handles quoted CSV fields correctly?

Enter a short answer in the next cell and run it. Retry after reviewing the
preceding examples if needed.


In [16]:
checkpoint_3_answer = ""  # enter your answer
check_answer(
    3, checkpoint_3_answer, 'csv',
    'The `csv` module handles delimiters and quoting safely.',
)


Checkpoint 3: enter your prediction, then run again.


False

---
## Part 7: Reading CSV Manually

Let's create a CSV file and parse it using `split(',')`.

**Figure 7.1: Creating a CSV file**

In [17]:
%%writefile students.csv
name,midterm,final
Elif Demir,85,90
Burak Yilmaz,92,88
Zeynep Kaya,78,82
Mert Ozturk,95,97
Ayse Celik,88,85

Writing students.csv


**Figure 7.2: Parsing a CSV file with `split(',')`**

In [18]:
with open("students.csv", "r") as file:
    header = file.readline().strip()  # Read the header line
    print("Columns:", header.split(","))
    print("-" * 40)
    
    for line in file:  # Read remaining lines
        line = line.strip()
        parts = line.split(",")
        
        name = parts[0]
        midterm = int(parts[1])
        final = int(parts[2])
        average = (midterm + final) / 2
        
        print(f"{name}: Midterm={midterm}, Final={final}, Average={average:.1f}")

Columns: ['name', 'midterm', 'final']
----------------------------------------
Elif Demir: Midterm=85, Final=90, Average=87.5
Burak Yilmaz: Midterm=92, Final=88, Average=90.0
Zeynep Kaya: Midterm=78, Final=82, Average=80.0
Mert Ozturk: Midterm=95, Final=97, Average=96.0
Ayse Celik: Midterm=88, Final=85, Average=86.5


**Figure 7.3: Storing CSV data in a list of lists**

In [19]:
data = []  # Will hold all rows

with open("students.csv", "r") as file:
    header = file.readline().strip().split(",")
    
    for line in file:
        row = line.strip().split(",")
        data.append(row)

print("Header:", header)
print("Data:")
for row in data:
    print(f"  {row}")

print(f"\nTotal students: {len(data)}")

Header:

 ['name', 'midterm', 'final']
Data:
  ['Elif Demir', '85', '90']
  ['Burak Yilmaz', '92', '88']
  ['Zeynep Kaya', '78', '82']
  ['Mert Ozturk', '95', '97']
  ['Ayse Celik', '88', '85']

Total students: 5


---
## Part 8: Writing CSV

To write CSV, we build strings with commas between values.

**Figure 8.1: Writing data as a CSV file**

In [20]:
# Data to write
cities = [
    ["Istanbul", 15000000, "Marmara"],
    ["Ankara", 5700000, "Central Anatolia"],
    ["Izmir", 4400000, "Aegean"],
    ["Bursa", 3100000, "Marmara"],
    ["Antalya", 2600000, "Mediterranean"]
]

with open("cities.csv", "w") as file:
    # Write header
    file.write("city,population,region\n")
    
    # Write each row
    for city in cities:
        line = f"{city[0]},{city[1]},{city[2]}\n"
        file.write(line)

# Verify
with open("cities.csv", "r") as file:
    print(file.read())

city,population,region
Istanbul,15000000,Marmara
Ankara,5700000,Central Anatolia
Izmir,4400000,Aegean
Bursa,3100000,Marmara
Antalya,2600000,Mediterranean



**Figure 8.2: Using `join()` to build CSV lines**

In [21]:
# A cleaner way using join()
header = ["product", "price", "quantity"]
products = [
    ["Notebook", "15.50", "100"],
    ["Pen", "3.25", "250"],
    ["Eraser", "1.75", "300"]
]

with open("products.csv", "w") as file:
    file.write(",".join(header) + "\n")  # Join with commas
    
    for product in products:
        file.write(",".join(product) + "\n")

# Verify
with open("products.csv", "r") as file:
    print(file.read())

product,price,quantity
Notebook,15.50,100
Pen,3.25,250
Eraser,1.75,300



---
## Part 9: Practical Example — Student Grades

Let's build a complete program that reads a grades CSV file, computes averages, determines pass/fail, and writes a summary.

**Figure 9.1: Creating the grades CSV data**

In [22]:
%%writefile grades.csv
name,hw1,hw2,hw3,midterm,final
Elif Demir,90,85,88,82,90
Burak Yilmaz,75,80,70,65,72
Zeynep Kaya,95,92,98,94,96
Mert Ozturk,60,55,58,45,50
Ayse Celik,85,88,82,78,80
Cem Arslan,70,65,72,68,74
Selin Tas,92,90,95,88,91

Writing grades.csv


**Figure 9.2: Complete grade processing program**

In [23]:
# Step 1: Read and parse the CSV
students = []

with open("grades.csv", "r") as file:
    header = file.readline().strip().split(",")
    
    for line in file:
        parts = line.strip().split(",")
        name = parts[0]
        scores = [int(x) for x in parts[1:]]  # Convert all scores to int
        students.append({"name": name, "scores": scores})

# Step 2: Compute averages and determine pass/fail
print(f"{'Name':<20} {'Average':>8} {'Status':>10}")
print("-" * 40)

results = []
for student in students:
    avg = sum(student["scores"]) / len(student["scores"])
    status = "PASS" if avg >= 60 else "FAIL"
    results.append((student["name"], avg, status))
    print(f"{student['name']:<20} {avg:>8.1f} {status:>10}")

# Step 3: Write summary to a new file
with open("grade_summary.csv", "w") as file:
    file.write("name,average,status\n")
    for name, avg, status in results:
        file.write(f"{name},{avg:.1f},{status}\n")

print("\n\u2705 Summary written to grade_summary.csv")

# Verify
print("\n--- grade_summary.csv ---")
with open("grade_summary.csv", "r") as file:
    print(file.read())

Name                  Average     Status
----------------------------------------
Elif Demir               87.0       PASS
Burak Yilmaz             72.4       PASS
Zeynep Kaya              95.0       PASS
Mert Ozturk              53.6       FAIL
Ayse Celik               82.6       PASS
Cem Arslan               69.8       PASS
Selin Tas                91.2       PASS

✅ Summary written to grade_summary.csv

--- grade_summary.csv ---
name,average,status
Elif Demir,87.0,PASS
Burak Yilmaz,72.4,PASS
Zeynep Kaya,95.0,PASS
Mert Ozturk,53.6,FAIL
Ayse Celik,82.6,PASS
Cem Arslan,69.8,PASS
Selin Tas,91.2,PASS



---
## Part 10: File Path Basics

When working with files, you need to know **where** the file is located.

### Relative vs Absolute Paths

| Path Type | Example | Description |
|:---|:---|:---|
| **Relative** | `"data.txt"` | In the current directory |
| **Relative** | `"data/grades.csv"` | In a subdirectory called `data` |
| **Absolute** | `"/home/user/data.txt"` | Full path from root |

### Google Colab File System

In Google Colab:
- Your **current directory** is `/content/`
- Files created with `%%writefile` or `open()` go to `/content/` by default
- Files are **temporary** — they disappear when the runtime disconnects
- You can mount Google Drive to save files permanently

**Figure 10.1: Checking the current directory and listing files**

In [24]:
import os

# Show current directory
print("Current directory:", os.getcwd())

# List files in current directory
print("\nFiles:")
for filename in os.listdir("."):
    if not filename.startswith("."):  # Skip hidden files
        print(f"  {filename}")

Current directory: [runtime folder]

Files:
  grades.csv
  output.txt
  students.csv
  greeting.txt
  fruits.txt
  demo.txt
  demo2.txt
  cities.csv
  students.txt
  grade_summary.csv
  products.csv


**Figure 10.2: Checking if a file exists before opening**

In [25]:
import os

filename = "greeting.txt"

if os.path.exists(filename):
    with open(filename, "r") as file:
        print(file.read())
else:
    print(f"File '{filename}' not found!")

Hello, welcome to Python!
File I/O is very useful.
This is the third line.
Learning to read files is fun.



> 💡 **Note:** Always check if a file exists before trying to read it, or use a `try/except` block to handle the `FileNotFoundError`.

---
### ⏱️ Checkpoint 4 of 5 — Paths (target 03:55)

Is an absolute path portable between computers? Answer yes or no.

Enter a short answer in the next cell and run it. Retry after reviewing the
preceding examples if needed.


In [26]:
checkpoint_4_answer = ""  # enter your answer
check_answer(
    4, checkpoint_4_answer, 'no',
    'Relative/configured paths are more portable.',
)


Checkpoint 4: enter your prediction, then run again.


False

---
## Exercises — Problems to Solve

> Each exercise is a problem. Think about the **STEPS** before you code. Decompose the problem, plan your approach, then implement it.

Complete the exercises below. Each exercise cell starts with `# ✏️ [EXn]` — do not remove that line.

### Core Practice and Optional Extension

- **Exercises 1–8:** core in-class practice.
- **Exercises 9 and above:** optional extension; these are not homework.
- At Checkpoint 5, list the exercises you have tested and can explain. This is a self-report, not automatic grading.


### Exercise 1: Create and Read (Easy)

Write a program that:
1. Creates a file called `"my_note.txt"` with 3 lines of your choice
2. Reads the file back and prints its contents

**Expected output (example):**
```
I love Python programming.
Files are really useful.
This is my third line.
```

<details><summary>💡 Hint</summary>

Use `open("my_note.txt", "w")` to write, then `open("my_note.txt", "r")` to read. Don't forget `\n` at the end of each line when writing.
</details>

In [27]:
# ✏️ [EX1]


### Exercise 2: Count Lines (Easy)

First run the cell below to create a file, then write a program that reads the file and counts how many lines it has.

**Expected output:**
```
The file has 5 lines.
```

<details><summary>💡 Hint</summary>

Use `readlines()` and `len()`, or use a counter variable in a `for` loop.
</details>

In [28]:
%%writefile poem.txt
Roses are red,
Violets are blue,
Python is great,
And so are you,
Happy coding!

Writing poem.txt


In [29]:
# ✏️ [EX2]


### Exercise 3: Write a List (Medium)

Given the list below, write each item to a file called `"colors.txt"`, one color per line. Then read the file and print it.

```python
colors = ["Red", "Blue", "Green", "Yellow", "Purple"]
```

**Expected output:**
```
Red
Blue
Green
Yellow
Purple
```

<details><summary>💡 Hint</summary>

Loop through the list and `write()` each color followed by `"\n"`. Or use `writelines()` with `[c + "\n" for c in colors]`.
</details>

In [30]:
# ✏️ [EX3]


### Exercise 4: Read Numbers (Medium)

First run the cell below to create a file of numbers. Then write a program that reads the file, converts each line to a number, and prints the **sum** and **average**.

**Expected output:**
```
Sum: 345
Average: 57.5
```

<details><summary>💡 Hint</summary>

Read each line, `strip()` it, convert to `int()`, and add to a list or a running total. The average is `sum / count`.
</details>

In [31]:
%%writefile numbers.txt
45
72
38
91
56
43

Writing numbers.txt


In [32]:
# ✏️ [EX4]


### Exercise 5: Copy File (Medium)

Write a program that reads `"poem.txt"` (created earlier) and writes its content to a new file called `"poem_copy.txt"`. Print a confirmation message.

**Expected output:**
```
File copied successfully!
Original: 5 lines
Copy: 5 lines
```

<details><summary>💡 Hint</summary>

Read the content of the source file with `read()`, then write it to the destination file with `write()`.
</details>

In [33]:
# ✏️ [EX5]


### Exercise 6: Search in File (Medium)

First run the cell below to create a file. Then write a program that asks for a search keyword and prints all lines containing that keyword (case-insensitive).

**Expected output (if keyword is "python"):**
```
Enter keyword: python
Found 2 matching lines:
  Line 1: Python is a popular programming language.
  Line 4: Many beginners choose Python as their first language.
```

<details><summary>💡 Hint</summary>

Use `if keyword.lower() in line.lower():` to do case-insensitive search. Keep a counter for line numbers.
</details>

In [34]:
%%writefile article.txt
Python is a popular programming language.
It was created by Guido van Rossum.
Java and C++ are also widely used.
Many beginners choose Python as their first language.
Programming skills are in high demand.

Writing article.txt


In [35]:
# ✏️ [EX6]


### Exercise 7: CSV Reader (Medium)

First run the cell below. Then write a program that reads the CSV file and prints the data in a formatted table.

**Expected output:**
```
Name                 Age    City           
--------------------------------------------
Elif Demir            21    Istanbul       
Burak Yilmaz          22    Ankara         
Zeynep Kaya           20    Izmir          
Mert Ozturk           23    Bursa          
Ayse Celik            21    Antalya        
```

<details><summary>💡 Hint</summary>

Read the header line first, then loop through the remaining lines. Use `split(",")` and f-string formatting with alignment (`:<20`, `:>5`).
</details>

In [36]:
%%writefile people.csv
name,age,city
Elif Demir,21,Istanbul
Burak Yilmaz,22,Ankara
Zeynep Kaya,20,Izmir
Mert Ozturk,23,Bursa
Ayse Celik,21,Antalya

Writing people.csv


In [37]:
# ✏️ [EX7]


### Exercise 8: CSV Writer (Medium)

Given the data below, write it to a CSV file called `"inventory.csv"` with a header row. Then read and print the file to verify.

```python
inventory = [
    ["Laptop", 5, 15000.00],
    ["Mouse", 25, 150.50],
    ["Keyboard", 15, 350.75],
    ["Monitor", 8, 4500.00]
]
```

**Expected output:**
```
product,quantity,price
Laptop,5,15000.0
Mouse,25,150.5
Keyboard,15,350.75
Monitor,8,4500.0
```

<details><summary>💡 Hint</summary>

Write the header first, then loop through the list. Use `f"{item[0]},{item[1]},{item[2]}\n"` or convert all values to strings and use `",".join()`.
</details>

In [38]:
# ✏️ [EX8]


---
### Checkpoint 5 of 5 — Practice reflection (target 04:45)

After Exercises 1–8, edit `practiced_exercises` in the next cell.
List only the exercise numbers whose results you have tested and whose steps you can explain.
Leave the list empty until you have done that work. This is your explicit self-report;
the tool does not inspect or grade your solution and does not count execution history.

**Türkçe:** Bu liste öz değerlendirmedir. Hücreyi çalıştırmak tek başına yeterli değildir;
sonucu kontrol et ve çözüm adımlarını açıklayabildiğinden emin ol.


In [ ]:
# Add an exercise number only after testing and explaining your own work.
# Example: [1, 2] records your reflection about Exercises 1 and 2.
# Türkçe: Bu liste öz değerlendirmedir; kodunuzun doğruluğunu otomatik ölçmez.
practiced_exercises = []
exercise_checkpoint(5, practiced_exercises, expected=8)
show_progress_summary()


---
## 🌟 Optional Extension

Exercises 9 and above are optional enrichment. Stop here if the five-hour class has ended.


### Exercise 9: Grade Report from CSV (Medium)

Read the `"grades.csv"` file (created in Part 9). For each student, compute the weighted average: **Homework 30%**, **Midterm 30%**, **Final 40%**. Print a report and write results to `"final_grades.csv"`.

The homework grade is the average of hw1, hw2, hw3.

**Expected output:**
```
Grade Report
============
Elif Demir       : HW=87.7 Mid=82 Fin=90 -> Weighted=86.9 PASS
Burak Yilmaz     : HW=75.0 Mid=65 Fin=72 -> Weighted=70.8 PASS
...
Results written to final_grades.csv
```

<details><summary>💡 Hint</summary>

Homework average = `(hw1 + hw2 + hw3) / 3`. Weighted = `hw_avg * 0.3 + midterm * 0.3 + final * 0.4`. Pass if weighted >= 60.
</details>

**Calculation check:** Elif: `(90+85+88)/3 = 87.666…`; `0.3*87.666… + 0.3*82 + 0.4*90 = 86.9`. Burak: `0.3*75 + 0.3*65 + 0.4*72 = 70.8`. Keep full precision until formatting. These weights belong to this sample data-processing exercise, not the CP1 course grading policy.


> **Example data only:** These fictional calculation weights are for this programming exercise. The course grade uses only the midterm (50%) and final (50%).


In [40]:
# ✏️ [EX9]


### Exercise 10: Temperature Log (Medium)

First run the cell below to create a temperature log. Then read the file and compute the **minimum**, **maximum**, and **average** temperatures. Print the results.

**Expected output:**
```
Temperature Report
==================
Readings: 7
Min: 18.5°C
Max: 31.2°C
Average: 24.0°C
```

<details><summary>💡 Hint</summary>

Skip the header line. Split each line by comma, convert the temperature (second column) to `float()`. Use `min()`, `max()`, and `sum()/len()` on the list of temperatures.
</details>

In [41]:
%%writefile temperatures.csv
date,temp_celsius
2024-01-01,22.5
2024-01-02,24.3
2024-01-03,19.8
2024-01-04,18.5
2024-01-05,31.2
2024-01-06,27.4
2024-01-07,24.1

Writing temperatures.csv


In [42]:
# ✏️ [EX10]


### Exercise 11: Append to Log (Medium)

Write a program that appends 3 new log entries to a file called `"app.log"`. Each entry should have a line number and a message. If the file already exists, new entries should be added to the end.

Run your code **twice** and verify that entries are appended (not overwritten).

**Expected output (after running twice):**
```
Log entries added!

Current log:
[1] Application started
[2] User logged in
[3] Data loaded successfully
[4] Application started
[5] User logged in
[6] Data loaded successfully
```

<details><summary>💡 Hint</summary>

First, read the existing file to count current entries (or start at 0 if file doesn't exist). Then open in append mode `"a"` and write new entries with line numbers continuing from the last entry.
</details>

In [43]:
# ✏️ [EX11]


### Exercise 12: Data Cleaner (Challenge)

First run the cell below to create a messy CSV file. Then write a program that:
1. Reads the file
2. Removes rows with empty values
3. Removes rows where age is not a valid number
4. Removes rows where age < 0 or age > 120
5. Writes the clean data to `"clean_data.csv"`
6. Prints how many rows were removed and why

**Expected output:**
```
Data Cleaning Report
====================
Total rows: 8
Removed: 3 rows
  - Row 3: empty value (age is empty)
  - Row 5: invalid age 'abc'
  - Row 7: age out of range (-5)
Clean rows: 5
Clean data written to clean_data.csv
```

<details><summary>💡 Hint</summary>

For each row, check: (1) does it have 3 values after split? (2) is the age field non-empty? (3) can it be converted to `int()`? (4) is it in the valid range? Use `try/except` for the conversion check.
</details>

In [44]:
%%writefile messy_data.csv
name,age,city
Elif Demir,21,Istanbul
Burak Yilmaz,22,Ankara
Zeynep Kaya,,Izmir
Mert Ozturk,23,Bursa
Ayse Celik,abc,Antalya
Cem Arslan,24,Trabzon
Selin Tas,-5,Eskisehir
Deniz Korkmaz,20,Konya

Writing messy_data.csv


In [45]:
# ✏️ [EX12]


### Bridge Exercise: Preview of Tables, NumPy and Plotting

Next week, you will load columns of numbers into **NumPy** arrays and draw them with **matplotlib**. Today, do the same job by hand.

Read `temperatures.csv` (written earlier in this notebook, in Exercise 10). Skip the header, collect the `temp_celsius` values in a list, and compute the mean with a loop.

**Expected output:**
```
Read 7 temperatures from temperatures.csv
Mean temperature: 23.97 °C
```

Next week `np.mean(temps)` gives the same number in one line, and `plt.plot(temps)` draws the readings.

<details><summary>💡 Hint</summary>

Open the file with `with`, call `f.readline()` once to skip the header, then loop over the remaining lines: `parts = line.strip().split(",")`, `value = float(parts[1])`, `total += value`, `count += 1`.
</details>


In [46]:
# ✏️ [EXBridge]


## Worked solutions and study support

Try each problem first. Then compare your reasoning and test cases with the complete
[Week 12 worked solutions](../solutions/Week_12_Solutions.ipynb).
The solution notebook includes every core exercise, optional exercise, and this week's bridge when present.

If you are using Colab, [open the published solution notebook](https://colab.research.google.com/github/ArifSolmaz/courses/blob/main/fall/cp1/solutions/Week_12_Solutions.ipynb).
Open it in a separate runtime. Running a solution first should not supply hidden variables to your own work.

**Türkçe:** Önce kendi çözümünü dene. Sonra adımları ve testleri karşılaştır; çözümü kapatıp farklı bir örneği kendin çöz.

[Simple course guide](../STUDY_GUIDE.md) · [All worked solutions](../solutions/README.md)
